# Roadmap GIS
**Đề tài:** Phân tích không gian & giá nhà —  phường trung tâm Hà Nội (Hoàn Kiếm, Ba Đình, Đống Đa)

Notebook này ghi lại từng bước đã thực hiện: lệnh/script dùng + hình kết quả tương ứng, theo đúng trình tự làm thực tế.
 


## Giai đoạn 1: Môi trường phát triển, Công nghệ & Công cụ sử dụng

Phần này tổng hợp toàn bộ ngăn xếp công nghệ (tech stack), nền tảng GIS, ngôn ngữ lập trình, hệ tọa độ và các thư viện chuyên dụng được sử dụng xuyên suốt chu trình thực hiện đồ án.

---

### 1.1. Ngôn ngữ & Môi trường phát triển
* **Python (v3.12+):** Ngôn ngữ lập trình cốt lõi cho việc thu thập dữ liệu (crawl), tiền xử lý bảng biểu, tính toán chỉ số không gian và mô hình hóa kinh tế lượng.
* **Jupyter Notebook (`.ipynb`):** Môi trường thực thi tương tác, ghi chép nhật ký phân tích, trực quan hóa kết quả và báo cáo theo từng giai đoạn.
* **Visual Studio Code / Antigravity IDE:** Môi trường phát triển tích hợp (IDE) quản lý mã nguồn, chạy script tự động hóa và cấu trúc thư mục dự án.

---

### 1.2. Phần mềm GIS & Nền tảng dữ liệu địa lý
* **QGIS (Quantum GIS v3.x):** 
  * Xử lý, biên tập và hiển thị dữ liệu không gian dạng vector và raster.
  * Thực thi các phân tích thống kê không gian chuyên sâu: Nội suy bề mặt giá (IDW), Phân tích điểm nóng (Getis-Ord $Gi^*$), Tự tương quan không gian cục bộ (Local Moran's I).
  * Thực hiện các phép truy vấn không gian phức tạp (Select by Expression, Select by Location) và ghép bảng thuộc tính (Spatial Join).
* **Overpass Turbo / OpenStreetMap (OSM API):** 
  * Công cụ truy vấn trực tuyến trên nền tảng bản đồ mở toàn cầu.
  * Khai thác ranh giới hành chính (`admin_level=6`), mạng lưới đường giao thông (`highway=*`) và 7 nhóm lớp tiện ích đô thị.
* **Chợ Tốt (chotot.com):** Nguồn dữ liệu bất động sản thứ cấp, cung cấp thông tin giá bán, diện tích, số tầng, loại hình nhà và tọa độ thực tế.

---

### 1.3. Hệ quy chiếu & Chuẩn dữ liệu địa không gian
* **WGS 84 (EPSG:4326):** Hệ tọa độ địa lý toàn cầu (kinh độ / vĩ độ), dùng làm chuẩn giao tiếp khi tải dữ liệu từ OpenStreetMap và định vị GPS từ tin rao bất động sản.
* **WGS 84 / UTM Zone 48N (EPSG:32648):** Hệ tọa độ phẳng chiếu theo lưới mét khu vực miền Bắc Việt Nam, bắt buộc sử dụng khi thực hiện các phép đo cự ly hình học (`cKDTree`), bán kính vùng đệm (Buffer 300m, 500m, 1,000m) và nội suy raster bề mặt để bảo đảm độ chính xác theo mét.
* **Định dạng dữ liệu lưu trữ:**
  * Vector: GeoJSON (`.geojson`), OGC GeoPackage (`.gpkg`).
  * Raster: GeoTIFF (`.tif`).
  * Bảng thuộc tính: CSV (`.csv`).

---

### 1.4. Danh mục thư viện Python chuyên dụng

| Phân nhóm nghiệp vụ | Tên thư viện | Phiên bản / Module chính | Vai trò & Ứng dụng cụ thể trong dự án |
|---|---|---|---|
| **Dữ liệu & Không gian** | `geopandas` | GeoDataFrame, I/O | Đọc/ghi các định dạng vector (`.geojson`, `.gpkg`), quản lý bảng thuộc tính không gian và chuyển đổi hệ quy chiếu (`to_crs`). |
| | `shapely` | `geometry.Point`, `Polygon` | Thao tác, khởi tạo và kiểm tra tính hợp lệ của các hình học vector. |
| | `fiona` / `pyproj` | Engine nền tảng | Hỗ trợ chuyển đổi toán học giữa các lưới chiếu tọa độ. |
| **Cấu trúc & Thuật toán** | `scipy` | `spatial.cKDTree` | Xây dựng cây tìm kiếm $k$-chiều để tối ưu hóa việc truy vấn điểm tiện ích gần nhất ($k=1$) cho từng căn nhà với độ phức tạp $O(\log N)$. |
| | | `query_ball_point` | Đếm nhanh số lượng tiện ích rơi vào các bán kính buffer hình cầu (300m, 500m, 1000m). |
| **Thu thập & Làm sạch** | `requests` | HTTP Client | Gửi request tải dữ liệu tin rao bất động sản từ API. |
| | `pandas` | DataFrame, Series | Làm sạch chuỗi văn bản, trích xuất số liệu, lọc trùng lặp (`drop_duplicates`), xử lý giá trị khuyết thiếu. |
| | `numpy` | Array, Matrix | Tính toán ma trận khoảng cách, vector hóa các phép biến đổi toán học. |
| **Thống kê & Machine Learning** | `statsmodels` | `formula.api.ols` | Huấn luyện mô hình định giá hưởng lạc (Hedonic Price Model), xuất báo cáo phân tích $R^2$, $F$-test, $p$-value và kiểm định đa cộng tuyến. |
| | `scikit-learn` | `RandomForestRegressor` | Huấn luyện mô hình học máy phi tuyến tính dự báo giá nhà từ tập biến không gian. |
| | | `KFold`, `cross_val_score` | Đánh giá độ tin cậy và hiện tượng quá khớp (overfitting) qua kỹ thuật kiểm định chéo 5-fold CV. |
| **Trực quan hóa đồ họa** | `matplotlib` | `pyplot` | Vẽ biểu đồ tán xạ phân tán (Actual vs Predicted), biểu đồ hộp (Boxplot) và xuất hình chuẩn in ấn. |
| | `seaborn` | Chart styling | Tinh chỉnh giao diện biểu đồ hệ số OLS, biểu đồ phân vị giá nhà theo từng địa bàn quận. |

---

### 1.5. Cấu trúc tổ chức thư mục dự án (Project Directory)

```text
Detai_GIS/
├── data/
│   ├── raw/                       # Dữ liệu gốc tải về (chưa qua xử lý)
│   │   ├── amenities_osm/         # 7 file GeoJSON tiện ích lấy từ Overpass Turbo
│   │   ├── boundary/              # Ranh giới 3 quận trung tâm (GeoJSON, GPKG)
│   │   ├── housing_listings/      # File CSV cào dữ liệu thô từ Chợ Tốt
│   │   └── region.png             # Ảnh vùng khoanh ranh giới tải dữ liệu
│   ├── interim/                   # Dữ liệu trung gian trong quá trình lọc
│   └── processed/                 # Dữ liệu sạch sau khi gắn thuộc tính không gian
│       └── housing_with_spatial_features.csv  # Bảng thuộc tính 121 dòng x 40 cột
├── QGIS_config/                          # Cấu hình dự án với QGIS
├── outputs/                       # Kết quả xuất ra từ các bước xử lý
│   ├── figures/                   # Biểu đồ mô hình (.png), bản đồ phân tích (.png)
│   └── query/                     # Ảnh chụp kết quả các thao tác truy vấn không gian
├── qgis_project/                  # File dự án QGIS (*.qgz / *.qgs) và layer GPKG
└── scripts/                       # Mã nguồn thực thi Python
    ├── 00_project_log.ipynb       # Notebook chính tích hợp toàn bộ nhật ký
    ├── 01_crawl_housing_3districts.py  # Script tự động cào dữ liệu tin đăng
    ├── 02_fix_boundary.py         # Script chuẩn hóa ranh giới hành chính
    ├── 03_spatial_analysis.py     # Script tính khoảng cách KDTree & đếm buffer
    ├── 04_hedonic_model.py        # Script chạy mô hình OLS & Random Forest
    └── 05_visualize_results.py    # Script xuất biểu đồ đánh giá mô hình

## Giai đoạn 2: Thu thập dữ liệu

### 2.1. Ranh giới hành chính + mạng lưới đường
Lấy qua Overpass Turbo (truy vấn `admin_level=6` cho 3 phường, và `highway=...` cho mạng lưới đường). Không dùng script Python cho bước này — thực hiện thủ công trên overpass-turbo.eu rồi xuất GeoJSON vào `data/raw/boundary/` và `data/raw/amenities_osm/roads_network.geojson`.

*(Lớp ranh giới không đưa vào các bản đồ trình bày cuối cùng, chỉ dùng nội bộ để khoanh vùng khi tải dữ liệu.)*

![vùng lấy data](../data/raw/region.png)

### 2.2. Tiện ích đô thị (OSM)
Tải qua Overpass Turbo theo từng loại: `amenity=school`, `amenity=hospital`, `leisure=park`, `shop=supermarket`, `shop=marketplace`, `highway=bus_stop`, `natural=water`. Mỗi loại lưu riêng 1 file `.geojson` trong `data/raw/amenities_osm/`.

#### 2.2.1. Trường học (`amenity=school`)
* **Số lượng thu thập:** 153 điểm.
* **Vai trò phân tích:** Đại diện cho khả năng tiếp cận dịch vụ giáo dục phổ thông. Khoảng cách đi bộ an toàn đến trường học là một trong những yếu tố gia tăng giá trị nhà đất mạnh nhất đối với phân khúc hộ gia đình.
![school](../outputs/figures/school.png)

#### 2.2.2. Cơ sở y tế & Bệnh viện (`amenity=hospital`)
* **Số lượng thu thập:** 54 điểm.
* **Vai trò phân tích:** Đo lường mức độ thuận tiện trong việc chăm sóc sức khỏe và tiếp cận cấp cứu y tế. Mật độ bệnh viện tuyến trung ương tại 3 quận lõi Hà Nội rất cao, tác động tích cực đến giá trị khai thác kinh doanh và dịch vụ lưu trú phụ trợ.
![hospital](../outputs/figures/hospital.png)

#### 2.2.3. Công viên & Không gian xanh (`leisure=park`)
* **Số lượng thu thập:** 89 điểm.
* **Vai trò phân tích:** Đại diện cho giá trị cảnh quan, vi khí hậu và chất lượng sống đô thị. Nhà gần công viên thường có giá bán cao nhờ không gian thoáng đãng và tiện ích rèn luyện sức khỏe.
![park](../outputs/figures/park.png)

#### 2.2.4. Siêu thị bán lẻ (`shop=supermarket`)
* **Số lượng thu thập:** 79 điểm.
* **Vai trò phân tích:** Phản ánh mức độ thương mại hóa hiện đại và tiện nghi sinh hoạt thường nhật của cư dân trong khu vực.
![supermarket](../outputs/figures/supermarket.png)

#### 2.2.5. Chợ dân sinh truyền thống (`shop=marketplace`)
* **Số lượng thu thập:** 39 điểm.
* **Vai trò phân tích:** Phản ánh đặc trưng văn hóa mua sắm truyền thống của cư dân phố thị Hà Nội. Tiện ích này vừa tạo thuận tiện sinh hoạt nhưng nếu quá sát nhà có thể gây ra ngoại ứng tiêu cực (tiếng ồn, ùn tắc giao thông).
![marketplace](../outputs/figures/marketplace.png)

#### 2.2.6. Trạm dừng xe buýt (`highway=bus_stop`)
* **Số lượng thu thập:** 462 điểm.
* **Vai trò phân tích:** Đo lường mật độ mạng lưới giao thông công cộng. Mật độ trạm xe buýt cao bảo đảm tính kết nối di chuyển nhưng cũng tiềm ẩn lưu lượng phương tiện đông đúc.
![bus_stop](../outputs/figures/bus_stop.png)

#### 2.2.7. Mặt nước & Hồ cảnh quan (`natural=water`)
* **Số lượng thu thập:** 124 điểm/vùng.
* **Vai trò phân tích:** Hà Nội có hệ thống hồ dày đặc (Hồ Gươm, Hồ Trúc Bạch, Hồ Ba Mẫu,...). Mặt nước đóng vai trò điều hòa vi khí hậu và là yếu tố tạo nên giá trị "view hồ" đắt đỏ bậc nhất trong định giá nhà đất.
![water](../outputs/figures/water.png)

### 2.3. Dữ liệu bất động sản (chotot)
Script: `scripts/crawl_housing_3districts.py` — cào dữ liệu theo từng quận, gộp với file cũ, lọc trùng, ghi vào `data/raw/housing_listings/housing_hanoi_center_150.csv`.

In [ ]:
# Trích đoạn từ scripts/crawl_housing_3districts.py (chỉ để tham khảo, không chạy trong notebook này)

# CẤU HÌNH QUẬN CẦN LẤY TIẾP THEO:
# Lần 1: Chạy lấy "Ba Đình"
# Lần 2: Sửa tên thành "Hoàn Kiếm" và chạy tiếp
# ...
# DISTRICT_NAME = "Hoàn Kiếm"   # Sửa thành "Hoàn Kiếm" sau khi Ba Đình xong
# AREA_ID = 12073               # Mã area_v2 của quận
# TARGET_PER_RUN = 50           # Lấy thêm 50 tin
#
# df_new = crawl_chotot(DISTRICT_NAME, AREA_ID, target_count=TARGET_PER_RUN)
# ... gộp với file cũ, lọc trùng theo id / (gia, dien_tich, dia_chi), ghi đè CSV

$ python scripts/03_spatial_analysis.py
Đọc 121 dòng, còn 121 dòng có tọa độ hợp lệ.
✓ bus_stop: 462 điểm -> đã tính khoảng cách + mật độ buffer
✓ hospital: 54 điểm -> đã tính khoảng cách + mật độ buffer
✓ marketplace: 39 điểm -> đã tính khoảng cách + mật độ buffer
✓ park: 89 điểm -> đã tính khoảng cách + mật độ buffer
✓ school: 153 điểm -> đã tính khoảng cách + mật độ buffer
✓ supermarket: 79 điểm -> đã tính khoảng cách + mật độ buffer
✓ water_lakes: 124 điểm -> đã tính khoảng cách + mật độ buffer

**Kết quả:** 121 dòng dữ liệu, đã có sẵn `latitude`/`longitude` (không cần geocode t).

```
Tổng số dòng dữ liệu hiện có: 121 dòng
Thống kê theo khu vực:
khu_vuc
Ba Đình      50
Đống Đa      45
Hoàn Kiếm    26
```

## Giai đoạn 3: Phân tích không gian

### 3.1. Tính biến không gian (khoảng cách + mật độ buffer)
Script thực thi: `scripts/03_spatial_analysis.py`
* **Hệ quy chiếu chuyển đổi:** Chuyển từ WGS84 (EPSG:4326) sang hệ tọa độ phẳng mét VN-2000 / UTM Zone 48N (EPSG:32648) để bảo đảm các phép đo khoảng cách và bán kính buffer tính chính xác theo mét.
* **Thuật toán xử lý:** 
  * Sử dụng cấu trúc cây `scipy.spatial.cKDTree` để truy vấn điểm tiện ích gần nhất ($k=1$) cho từng căn nhà, sinh ra các biến khoảng cách `dist_{amenity}_m`.
  * Dùng `query_ball_point` để đếm số lượng tiện ích nằm trong các vòng đệm bán kính 300m, 500m và 1000m, sinh ra các biến mật độ `count_{amenity}_{radius}m`.
* **Kết quả xử lý:** Hoàn thành tính toán cho 121 căn nhà hợp lệ trên 7 lớp tiện ích, xuất file tổng hợp `data/processed/housing_with_spatial_features.csv` (121 dòng $\times$ 40 cột).

In [ ]:
# Trích đoạn từ scripts/03_spatial_analysis.py 

# for file in amenity_files:
#     amenity_type = os.path.splitext(os.path.basename(file))[0]
#     gdf_amenity = gpd.read_file(file).to_crs(CRS_METRIC)
#     tree = cKDTree(coords_amenity)
#     dist, _ = tree.query(coords_housing, k=1)
#     gdf_housing[f"dist_{amenity_type}_m"] = dist
#     for radius in BUFFER_RADII:
#         counts = tree.query_ball_point(coords_housing, r=radius, return_length=True)
#         gdf_housing[f"count_{amenity_type}_{radius}m"] = counts

**Kết quả console:**
```
Đọc 121 dòng, còn 121 dòng có tọa độ hợp lệ.
✓ bus_stop, hospital, marketplace, park, school, supermarket, water_lakes -> đã tính khoảng cách + mật độ buffer
Đã lưu: data/processed/housing_with_spatial_features.csv
Số dòng: 121, số cột: 40
```
File output: `data/processed/housing_with_spatial_features.csv`

### 3.2. Nội suy bề mặt giá (Inverse Distance Weighting - IDW)
Công cụ QGIS: `Processing Toolbox > Interpolation > IDW`

![IDW interpolation](../outputs/figures/map_idw_price.png)

> **Thông số kỹ thuật & Thiết lập:**
> * **Lớp điểm đầu vào:** `housing_utm` (121 điểm khảo sát bất động sản).
> * **Thuộc tính nội suy:** `gia_trieu_m2` (Đơn giá tính theo triệu VNĐ/m²).
> * **Bậc suy giảm khoảng cách (Distance Coefficient $P$):** $P = 2$ (mô hình IDW bậc 2 chuẩn).
> * **Phạm vi nội suy (Extent):** Cắt theo mặt nạ đệm bao quanh 3 quận trung tâm (`Clipped (mask)`).

* **Định nghĩa & Cơ sở lý thuyết:**  
  IDW là phương pháp nội suy tiền định dựa trên Định luật thứ nhất của Địa lý học (Tobler's First Law of Geography): *"Mọi sự vật đều liên quan đến nhau, nhưng những sự vật ở gần nhau thì liên quan nhiều hơn những sự vật ở xa"*. Giá trị giá nhà tại một điểm chưa biết tọa độ $x$ được tính bằng trung bình trọng số của các điểm khảo sát lân cận:
  $$\hat{Z}(x) = \frac{\sum_{i=1}^n \frac{1}{d_i^p} Z_i}{\sum_{i=1}^n \frac{1}{d_i^p}}$$
  Trong đó $d_i$ là khoảng cách từ điểm cần ước lượng đến điểm khảo sát thứ $i$, và $p$ là hệ số lũy thừa khoảng cách.

* **Ý nghĩa & Diễn giải bản đồ:**  
  * **Vùng đỉnh giá (Màu đỏ - cam):** Tập trung đậm đặc tại khu vực lõi lịch sử quận Hoàn Kiếm (khu phố cổ, quanh Hồ Gươm) và dải phía Đông quận Ba Đình. Tại đây, mức giá đơn vị vượt ngưỡng 300 - 450+ triệu VNĐ/m².
  * **Vùng chuyển tiếp (Màu vàng - xanh lá nhạt):** Phân bố tại khu vực tiếp giáp giữa Ba Đình và Đống Đa, mức giá dao động phổ biến từ 180 - 250 triệu VNĐ/m².
  * **Vùng trũng giá tương đối (Màu xanh lam - xanh ngọc):** Phân bố rộng ở nửa phía Tây và phía Nam quận Đống Đa (quanh trục Xã Đàn kéo dài, Kim Liên, Thái Hà), giá tập trung ở mức 120 - 180 triệu VNĐ/m².

* **Tác dụng & Ảnh hưởng:**  
  * Chuyển đổi dữ liệu rời rạc 121 điểm thành bề mặt liên tục, giúp quan sát ngay lập tức cấu trúc không gian của thị trường bất động sản trung tâm.

### 3.3. Phân tích điểm nóng (Hot Spot Analysis - Getis-Ord Gi*)
Công cụ QGIS: `Processing Toolbox > Spatial Statistics > Getis-Ord Gi*`

![Hot Spot Analysis](../outputs/figures/GI_output.png)

> **Bảng thống kê kết quả phân nhóm điểm ($N = 121$):**
> 
> | Phân loại cụm | Mức ý nghĩa thống kê | Ngưỡng $z$-score / $p$-value | Số lượng điểm | Tỷ lệ (%) |
> |---|---|---|---|---|
> | **Hotspot 99% confidence** | Cực nóng (giá cao tập trung) | $z > +2.58 \ (p < 0.01)$ | **26** | 21.49% |
> | **Hotspot 95% confidence** | Nóng vừa | $z > +1.96 \ (p < 0.05)$ | **3** | 2.48% |
> | **Hotspot 90% confidence** | Nóng nhẹ | $z > +1.645 \ (p < 0.10)$ | **4** | 3.31% |
> | **Not Significant** | Không có ý nghĩa cụm | $-1.645 \le z \le +1.645$ | **55** | 45.45% |
> | **Coldspot 90% confidence** | Lạnh nhẹ | $z < -1.645 \ (p < 0.10)$ | **0** | 0.00% |
> | **Coldspot 95% confidence** | Lạnh vừa | $z < -1.96 \ (p < 0.05)$ | **1** | 0.83% |
> | **Coldspot 99% confidence** | Cực lạnh (giá thấp tập trung) | $z < -2.58 \ (p < 0.01)$ | **32** | 26.45% |

* **Định nghĩa & Cơ sở lý thuyết:**  
  Chỉ số Getis-Ord $G_i^*$ đánh giá sự tập hợp không gian của các giá trị cao hoặc thấp. Một điểm có giá trị giá bán cao không nhất thiết là một Hot Spot; điểm đó chỉ trở thành Hot Spot khi bản thân nó mang giá trị cao và được bao quanh bởi các điểm lân cận cũng có giá trị cao vượt trội so với phân phối ngẫu nhiên toàn cục:
  $$G_i^* = \frac{\sum_{j=1}^n w_{ij} x_j - \bar{X} \sum_{j=1}^n w_{ij}}{S \sqrt{\frac{n \sum_{j=1}^n w_{ij}^2 - (\sum_{j=1}^n w_{ij})^2}{n - 1}}}$$

* **Ý nghĩa & Diễn giải bản đồ:**  
  * **Cực điểm nóng (Hotspot 99% - 26 điểm đỏ):** Tập trung hoàn toàn ở quận Hoàn Kiếm và một phần giáp ranh Ba Đình. Đây là trung tâm hành chính - du lịch - thương mại dịch vụ cao cấp nhất Thủ đô, nơi quỹ đất cực kỳ khan hiếm.
  * **Cực điểm lạnh (Coldspot 99% - 32 điểm xanh tím):** Phân cụm rõ rệt ở khu vực quận Đống Đa. "Lạnh" ở đây mang tính so sánh nội bộ: trong phạm vi 3 quận nội thành lõi, khu vực Đống Đa có mức giá trên mỗi $m^2$ thấp hơn đáng kể so với khu phố cổ Hoàn Kiếm.
  * **Điểm trung tính (Not significant - 55 điểm xám rỗng):** Chiếm đa số tại quận Ba Đình, cho thấy tại đây giá nhà đất có độ biến động cao và phân tán đều, không hình thành các cụm tập trung cực đoan.

* **Tác dụng & Ảnh hưởng:**  
  * Khẳng định tính phân cực không gian mạnh mẽ của thị trường nhà đất nội đô: Phân tách rõ ràng 2 cực giá trị (Hoàn Kiếm đối trọng với Đống Đa).
  * Làm cơ sở để đưa biến phân nhóm cụm không gian vào mô hình hồi quy giá đất nhằm kiểm soát hiệu ứng tự tương quan không gian.

### 3.4. Tự tương quan không gian cục bộ (Local Moran's I - LISA)
Công cụ QGIS: `Processing Toolbox > Univariate Local Moran's I`

![Local Moran's I](../outputs/figures/moran_output.png)

> **Bảng thống kê phân nhóm phần tử LISA ($N = 121$):**
> 
> | Loại hình liên kết | Diễn giải thuộc tính không gian | Số lượng điểm | Tỷ lệ (%) |
> |---|---|---|---|
> | **HH (High - High)** | Cụm giá cao nằm cạnh các vùng giá cao | **19** | 15.70% |
> | **LL (Low - Low)** | Cụm giá thấp nằm cạnh các vùng giá thấp | **32** | 26.45% |
> | **LH (Low - High)** | Ngoại lai: Điểm giá thấp lọt giữa vùng giá cao | **17** | 14.05% |
> | **HL (High - Low)** | Ngoại lai: Điểm giá cao vượt trội giữa vùng giá thấp | **15** | 12.40% |
> | **Not Significant** | Không có tương quan không gian cục bộ | **38** | 31.40% |

* **Định nghĩa & Cơ sở lý thuyết:**  
  Chỉ số Local Moran's I đo lường mức độ tương đồng hoặc dị biệt giữa giá trị của một căn nhà với các căn nhà lân cận trong ma trận không gian:
  $$I_i = \frac{x_i - \bar{X}}{S_i^2} \sum_{j=1, j \neq i}^n w_{ij} (x_j - \bar{X})$$
  Khác với Getis-Ord Gi* chỉ phát hiện cụm giá trị cùng cao hoặc cùng thấp, Local Moran's I có năng lực phân tách các **điểm dị biệt không gian (Spatial Outliers)** gồm nhóm **HL** và **LH**.

* **Ý nghĩa & Diễn giải bản đồ:**  
  * **Cụm đồng nhất HH (19 điểm đỏ):** Lõi bất động sản cao cấp tại quận Hoàn Kiếm, nơi toàn bộ các bất động sản xung quanh đều giữ mức giá đỉnh.
  * **Cụm đồng nhất LL (32 điểm xanh lam):** Cụm nhà ở dân cư truyền thống tại Đống Đa với mặt bằng giá thấp đồng đều.
  * **Điểm dị biệt LH (17 điểm xanh dương nhạt):** Nằm xen kẽ trong khu vực Hoàn Kiếm và Đông Ba Đình. Đây là các bất động sản giá rẻ nằm giữa khu "đất vàng" — phản ánh thực tế nhà trong ngõ sâu, ngõ cụt xe máy không vào được, nhà diện tích siêu nhỏ hoặc vướng tranh chấp pháp lý/sổ chung.
  * **Điểm dị biệt HL (15 điểm hồng cam):** Nằm rải rác giữa quận Đống Đa. Đây là các bất động sản có giá vượt trội nằm giữa khu vực giá bình dân — đại diện cho các căn nhà mặt phố thương mại lớn (Thái Hà, Chùa Bộc, Xã Đàn) hoặc biệt thự liền kề phân lô cao cấp.

* **Tác dụng & Ảnh hưởng:**  
  * Cực kỳ quan trọng cho công tác thẩm định giá và lọc sạch dữ liệu: Giúp mô hình nhận diện chính xác các bất động sản có đặc tính cá biệt mà nếu chỉ nhìn tọa độ sẽ bị định giá sai lệch.
  * Hạn chế sai số phương sai trong các mô hình kinh tế lượng không gian (Spatial Error / Spatial Lag Model).

### 3.5. Truy vấn không gian

#### a) Tổng thể phân bố mẫu khảo sát
![housing_full](../outputs/query/housing_full.png)

> **Mô tả hiện trạng lớp dữ liệu:**
> Toàn bộ 121 điểm bất động sản sau khi làm sạch tọa độ, trải đều trên địa bàn 3 quận lõi Hà Nội: Ba Đình (50 tin), Đống Đa (45 tin), Hoàn Kiếm (26 tin). Lớp dữ liệu đóng vai trò nền tảng để thực hiện các phép lọc thuộc tính và phân tích khoảng cách lân cận.

#### b) Truy vấn thuộc tính (Select by Expression)
Công cụ QGIS: `Select features by expression`  
Cú pháp thực hiện: `"gia_trieu_m2" > 250 OR "dist_school_m" < 300`

![Select by Expression](../outputs/query/filter_gia.png)

> **Thông số kết quả truy vấn:**
> * **Số lượng bản ghi thỏa mãn:** **104 / 121** bất động sản (chiếm **85.95%** toàn bộ tập mẫu).
> * **Ký hiệu hiển thị:** Các điểm được chọn chuyển sang màu vàng viền chéo trên bản đồ.

* **Ý nghĩa & Cơ sở thiết lập điều kiện:**  
  * Điều kiện `"gia_trieu_m2" > 250`: Lọc nhóm bất động sản thuộc phân khúc cao cấp và siêu sang của vùng trung tâm.
  * Điều kiện `"dist_school_m" < 300`: Lọc các căn nhà có khả năng tiếp cận trường học trong bán kính đi bộ lý tưởng (dưới 5 phút đi bộ của học sinh và phụ huynh).
  * Phép toán logic `OR`: Bao hàm cả 2 nhóm nhu cầu quan trọng nhất của người mua nhà tại vùng lõi: Nhà có giá trị kinh tế/thương mại cao HOẶC nhà có lợi thế sinh hoạt gia đình vượt trội.
* **Tác dụng:** Giúp người mua hoặc nhà đầu tư khoanh vùng nhanh chóng 86% quỹ căn phù hợp với tiêu chí ưu tiên chất lượng vị trí hoặc phân khúc tài chính.

#### c) Truy vấn không gian theo cự ly (Select by Location / Select within distance)
Công cụ QGIS: `Processing Toolbox > Vector Selection > Select within distance`

![Select by Location](../outputs/query/filter_location.png)

> **Thông số thiết lập kỹ thuật:**
> * **Lớp đối tượng được lọc (Input Layer):** `housing_utm` [EPSG:32648].
> * **Lớp đối tượng so sánh (Comparing Layer):** `hospital` [EPSG:4326] (54 cơ sở y tế).
> * **Ngưỡng khoảng cách tối đa (Distance Threshold):** **500.0 mét**.
> * **Kết quả lựa chọn:** **59 / 121** căn nhà thỏa mãn điều kiện (chiếm **48.76%**).

* **Ý nghĩa & Diễn giải phân bố:**  
  * Cự ly 500m là cự ly đi bộ chuẩn quốc tế (Pedestrian Catchment Area - khoảng 6 đến 7 phút di chuyển).
  * Kết quả chỉ ra gần một nửa số lượng nhà ở khảo sát tại 3 quận trung tâm được tiếp cận trực tiếp với bệnh viện tuyến quận/trung ương trong cự ly đi bộ. Sự hiện diện dày đặc của các điểm được chọn ở Ba Đình và Hoàn Kiếm chứng minh lợi thế hạ tầng xã hội hoàn thiện vượt bậc của khu vực trung tâm Hà Nội so với các vùng ngoại vi.
* **Tác dụng:** Cung cấp thuộc tính nhị phân (`is_near_hospital_500m`) làm biến độc lập bổ trợ giải thích sự chênh lệch giá thuê hoặc giá bán nhà ở.

#### d) Ghép thuộc tính không gian (Spatial Join)
Công cụ QGIS: `Vector > Data Management Tools > Join Attributes by Location`

![Spatial Join](../outputs/query/spatial_join.png)

> **Quy trình tích hợp:**
> Thực hiện gán nhãn thuộc tính phân cụm từ các kết quả phân tích thống kê không gian (nhãn Hotspot/Coldspot của Gi* và phân loại HH/LL/HL/LH của Moran's I) ngược trở lại bảng thuộc tính của từng điểm nhà ở `housing_utm`.

* **Tác dụng & Ý nghĩa trong quy trình GIS:**  
  * Tạo bảng dữ liệu hợp nhất hoàn chỉnh: Mỗi dòng nhà ở giờ đây vừa có thông tin nội tại (giá, diện tích, số tầng), vừa có khoảng cách đến 7 loại tiện ích, vừa mang thông tin phân vùng cụm không gian.
  * Là bước tiền xử lý bắt buộc để xuất file dữ liệu phục vụ huấn luyện mô hình dự báo giá nhà (Hedonic Price Model / OLS Regression / Random Forest).

## Giai đoạn 4: Mô hình hedonic 

### Kết quả OLS (n = 121, R² = 0.407, Adj. R² = 0.322, p(F) = 5.58e-07)

| Biến | Hệ số | p-value | Ý nghĩa |
|---|---|---|---|
| dist_water_lakes_m | -0.209 | **0.047** | Có ý nghĩa — mỗi mét xa hồ nước, giá giảm ~0.21 triệu/m² |
| khu_vuc_Hoàn Kiếm | +202.39 | **0.011** | Có ý nghĩa — nhà ở Hoàn Kiếm đắt hơn Ba Đình (nhóm tham chiếu) ~202 triệu/m² |
| loai_nha_Mặt phố | +87.99 | **0.018** | Có ý nghĩa — nhà mặt phố đắt hơn nhà hẻm ~88 triệu/m² |
| dien_tich, so_tang | +0.40 / +8.77 | 0.174 / 0.231 | Không có ý nghĩa thống kê ở mức dữ liệu hiện tại |
| dist_school/hospital/park/supermarket/bus_stop/marketplace_m | — | đều > 0.13 | Không có ý nghĩa — khả năng vì `khu_vuc` đã gánh phần lớn hiệu ứng vị trí |
| Z_score, p_value, p_fdr | — | 0.42–0.60 | Không có ý nghĩa — và gây cảnh báo đa cộng tuyến (Cond. No. = 8.26e+04) vì 3 biến này tính từ cùng 1 thống kê Local Moran's I |

**Diễn giải:** Mô hình giải thích được ~41% biến thiên giá nhà/m², mức khá ổn với dữ liệu tin rao vặt (nhiễu do giá người bán tự đưa ra). Hai yếu tố có ảnh hưởng rõ nhất và nhất quán với lý thuyết hedonic là **vị trí khu vực** (Hoàn Kiếm) và **loại nhà** (mặt phố/hẻm). Đa số biến khoảng cách tiện ích không có ý nghĩa riêng lẻ — hợp lý vì biến `khu_vuc` (3 phường trung tâm, vốn đã rất gần nhau) có thể đã hấp thụ phần lớn hiệu ứng vị trí, khiến các biến khoảng cách chi tiết hơn khó tách biệt được đóng góp riêng với cỡ mẫu 121.

**Hạn chế cần nêu trong báo cáo:** 3 biến `Z_score/p_value/p_fdr` (từ Local Moran's I) được tính dựa trên chính giá nhà, nên đưa vào làm biến dự báo mang tính minh họa mối liên hệ hơn là quan hệ nhân quả — đây cũng là nguồn gây cảnh báo đa cộng tuyến trong bảng kết quả.

### So sánh với Random Forest

| Mô hình | R² | Ghi chú |
|---|---|---|
| OLS | 0.407 | Dễ diễn giải hệ số, nhưng giả định tuyến tính |
| Random Forest (5-fold CV) | 0.092 (±0.346) | Dao động lớn giữa các fold — dấu hiệu overfit do 36 biến/121 quan sát, không phải lỗi mô hình |

![So sánh R²](../outputs/figures/model_r2_comparison.png)

Dù R² của Random Forest thấp hơn, biến quan trọng nhất theo RF (`count_supermarket_1000m`, mật độ siêu thị trong 1km) khá nhất quán giữa các lần chạy — gợi ý rằng **mật độ tiện ích xung quanh** có thể là một dạng quan hệ phi tuyến mà OLS không bắt được tốt.

### 4.1. Phân tích hệ số hồi quy OLS (Hedonic Price Model)

![Hệ số OLS](../outputs/figures/ols_coefficients.png)

> **Thông số kỹ thuật & Thiết lập mô hình:**
> * **Biến phụ thuộc ($Y$):** `gia_trieu_m2` (Đơn giá tính theo triệu VNĐ/m²).
> * **Quy tắc phân loại màu sắc:** 
>   * Cột màu đỏ: Biến có ý nghĩa thống kê ở mức $\alpha = 0.05$ ($p < 0.05$).
>   * Cột màu xám: Biến không có ý nghĩa thống kê tin cậy ($p \ge 0.05$).
> * **Nhóm chuẩn tham chiếu (Reference Category):** Quận Ba Đình (đối với biến khu vực) và Nhà ngõ/hẻm (đối với loại nhà).

* **Ý nghĩa & Diễn giải định lượng từng biến số:**
  * **Hiệu ứng vị trí trung tâm (`khu_vuc_Hoàn Kiếm`):** Hệ số đạt **$+202.39$ triệu VNĐ/m²** ($p = 0.011 < 0.05$). Khi các yếu tố về diện tích, số tầng và tiện ích xung quanh tương đương nhau, một bất động sản tại Hoàn Kiếm có giá trị trung bình cao hơn khoảng 202 triệu VNĐ/m² so với Ba Đình. Điều này phản ánh giá trị thặng dư thương mại và tính khan hiếm tuyệt đối của quỹ đất khu vực phố cổ - Hồ Gươm.
  * **Hiệu ứng loại hình mặt tiền (`loai_nha_Mặt phố`):** Hệ số đạt **$+87.99$ triệu VNĐ/m²** ($p = 0.018 < 0.05$). Nhà mặt phố có giá cao hơn khoảng 88 triệu VNĐ/m² so với nhà trong ngõ/hẻm nhờ tiềm năng khai thác kinh doanh trực tiếp và tính thanh khoản cao.
  * **Hiệu ứng cự ly cảnh quan (`dist_water_lakes_m`):** Hệ số **$-0.209$** ($p = 0.047 < 0.05$), mang dấu âm phù hợp với lý thuyết kinh tế đô thị: mỗi mét cách xa hồ nước khiến đơn giá nhà giảm khoảng 0.21 triệu VNĐ/m² (tương đương giảm ~210,000 VNĐ/m² cho mỗi mét khoảng cách).
  * **Các biến khoảng cách tiện ích khác (`dist_*_m`):** Hầu hết đều có hệ số rất nhỏ và không đạt mức ý nghĩa thống kê ($p > 0.13$). Lý do là tại vùng lõi 3 quận nội thành, mạng lưới trường học, trạm y tế, siêu thị, chợ dân sinh và điểm dừng xe buýt đã phân bố rất dày đặc; đồng thời biến định tính `khu_vuc` đã hấp thụ phần lớn phương sai không gian toàn cục.
  * **Hiện tượng đa cộng tuyến nghiêm trọng (`p_value`, `p_fdr`):** Hai thanh màu xám vọt lên $+720$ và rơi xuống $-690$ là dấu hiệu điển hình của đa cộng tuyến (Condition Number chạm $8.26 \times 10^4$). Do các biến $Z$-score, $p$-value và $p_{FDR}$ được tính toán phái sinh trực tiếp từ chính giá nhà qua thống kê Local Moran's I, việc đưa đồng thời chúng vào hàm hồi quy tuyến tính đã làm nhiễu loạn ma trận hiệp phương sai.

### 4.2. Đánh giá tương quan giá thực tế vs dự đoán

![Giá thực tế vs dự đoán](../outputs/figures/actual_vs_predicted.png)

> **Thông số kiểm định mô hình OLS:**
> * **Quy mô mẫu:** $N = 121$ quan sát bất động sản hợp lệ.
> * **Hệ số xác định ($R^2$):** **$0.407$** (Mô hình giải thích được $40.7\%$ biến thiên của đơn giá/m²).
> * **Hệ số xác định hiệu chỉnh (Adjusted $R^2$):** **$0.322$**.
> * **Kiểm định độ phù hợp chung:** $p(F) = 5.58 \times 10^{-7}$ (Mô hình có ý nghĩa thống kê tổng quát rất cao).
> * **Đường nét đứt màu đỏ ($y = x$):** Đường tham chiếu dự đoán hoàn hảo (giá dự đoán bằng đúng giá thực tế).

* **Ý nghĩa & Nhận xét phân bố sai số:**
  * **Vùng dự báo hội tụ tốt (150 – 400 triệu VNĐ/m²):** Các điểm phân tán tập trung dày đặc và bám sát quanh trục đường chéo đỏ. Đây là dải giá phổ biến của nhà ở dân cư tại Ba Đình và Đống Đa, nơi hàm OLS tuyến tính phát huy độ chuẩn xác cao nhất.
  * **Ước lượng vượt mức ở nhóm giá thấp (< 200 triệu VNĐ/m²):** Các điểm dữ liệu nằm lệch lên *phía trên* đường tham chiếu đỏ. Đối với những căn nhà có giá rẻ nằm sâu trong ngõ hẹp hoặc vướng mắc pháp lý, do mô hình OLS chưa có biến đo lường chiều rộng ngõ trước nhà nên có xu hướng dự đoán mức giá cao hơn thực tế.
  * **Hiện tượng "trần dự báo" ở phân khúc siêu sang (> 600 triệu VNĐ/m²):** Hầu hết các điểm khảo sát giá cao (từ 600 đến gần 1,200 triệu VNĐ/m²) đều rơi xuống *phía dưới* đường đỏ. Mức giá dự đoán tối đa của mô hình OLS chỉ chạm ngưỡng ~650 – 800 triệu VNĐ/m².
  * **Hạn chế kỹ thuật:** Sự nén ép phương sai này phản ánh hiện tượng quy chuẩn về trung bình (Regression to the mean) của phương pháp OLS truyền thống trên tập dữ liệu lệch phải (right-skewed), gợi mở sự cần thiết của các phép biến đổi logarit ($\ln(Price)$) hoặc các mô hình kinh tế lượng không gian phi tuyến.

### 4.3. Phân bố biến động giá theo địa bàn quận (Boxplot)

![Phân bố giá theo khu vực](../outputs/figures/price_by_district_boxplot.png)

> **Thống kê mô tả cấu trúc phân vị ($N = 121$):**
> * **Quận Hoàn Kiếm ($n = 26$):** 
>   * Trung vị (đường xanh lá): $\approx 502$ triệu VNĐ/m².
>   * Khoảng trải giữa ($IQR$): Rộng nhất trong 3 quận ($\sim 300 - 640$ triệu VNĐ/m²).
>   * Điểm ngoại lai cực đại: Xuất hiện 1 bất động sản có giá xấp xỉ **1,185 triệu VNĐ/m²**.
> * **Quận Ba Đình ($n = 50$):** 
>   * Trung vị: $\approx 290$ triệu VNĐ/m².
>   * Khoảng trải giữa ($IQR$): Dao động ổn định trong khoảng $\sim 215 - 400$ triệu VNĐ/m².
>   * Điểm ngoại lai: Xuất hiện 1 điểm đơn lẻ tại mức $\approx 680$ triệu VNĐ/m².
> * **Quận Đống Đa ($n = 45$):** 
>   * Trung vị: Thấp nhất trong 3 quận, đạt $\approx 208$ triệu VNĐ/m².
>   * Khoảng trải giữa ($IQR$): Tập trung cao trong khoảng $\sim 160 - 285$ triệu VNĐ/m².
>   * Điểm ngoại lai: Xuất hiện **3 điểm ngoại lai** vượt trội ở dải giá $\sim 480 - 555$ triệu VNĐ/m².

* **Ý nghĩa & Diễn giải phân hóa không gian:**
  * **Sự vượt trội của địa tô Hoàn Kiếm:** Hộp phân vị của Hoàn Kiếm nằm tách biệt hoàn toàn ở tầng giá cao. Biên độ phân tán rất rộng phản ánh sự chênh lệch lớn giữa nhà mặt phố thương mại sầm uất và nhà nằm trong các ngõ ngách bảo tồn cổ kính.
  * **Tính đồng nhất của phân khúc nhà ở Đống Đa:** Mức giá tại Đống Đa có trung vị thấp nhất và cấu trúc nén chặt nhất, đại diện cho khu vực phát triển nhà ở dân cư truyền thống và các khu tập thể có mặt bằng giá ổn định.
  * **Sự tương quan chặt chẽ với các chỉ số thống kê không gian (QGIS):**
    * Mặt bằng giá đỉnh tại Hoàn Kiếm giải thích trực tiếp việc hình thành cụm **Hotspot 99%** (Getis-Ord $Gi^*$) và cụm **High-High** (Local Moran's I).
    * Mặt bằng giá thấp đồng đều tại Đống Đa tương ứng trực tiếp với vùng cụm **Coldspot 99%** và cụm **Low-Low**.
    * Đáng chú ý, **3 điểm ngoại lai giá cao tại Đống Đa** (các vòng tròn đơn độc nằm trên râu hộp) chính là các vị trí được mô hình Local Moran's I định danh là điểm dị biệt không gian **HL (High-Low Outlier)**: Những căn nhà mặt phố thương mại lớn (Chùa Bộc, Thái Hà, Xã Đàn) sở hữu giá trị vượt bậc so với nền giá chung của các khu dân cư lân cận.

## Giai đoạn 5: Tổng kết toàn diện đồ án

Đồ án đã xây dựng hoàn chỉnh một quy trình khoa học dữ liệu không gian tích hợp (Spatial Data Science Pipeline) khép kín, từ khâu trích xuất dữ liệu thô, phân tích cấu trúc địa lý đến mô hình hóa lượng giá bất động sản tại 3 quận trung tâm Hà Nội (Ba Đình, Hoàn Kiếm, Đống Đa).

---

### 5.1. Tổng kết quy trình kỹ thuật & Thành quả xử lý GIS
1. **Thu thập & Chuẩn hóa dữ liệu đa nguồn:**
   * Trích xuất thành công ranh giới hành chính, mạng lưới giao thông và **1,000+ đối tượng tiện ích đô thị** qua Overpass API (OpenStreetMap) thuộc 7 phân nhóm thiết yếu: trường học (153), y tế (54), công viên (89), siêu thị (79), chợ (39), trạm bus (462) và mặt nước (124).
   * Cào và làm sạch tọa độ địa lý cho **121 tin rao bất động sản thực tế**, bảo đảm tính đại diện trên cả 3 địa bàn quận (Ba Đình: 50, Đống Đa: 45, Hoàn Kiếm: 26).
2. **Kỹ thuật tính toán biến không gian ở quy mô lớn:**
   * Đồng bộ hệ quy chiếu dự đoán tọa độ phẳng VN-2000 / UTM Zone 48N (EPSG:32648) để duy trì tính toàn vẹn khoảng cách hệ mét.
   * Ứng dụng cấu trúc dữ liệu cây $k$-chiều (`cKDTree`) và thuật toán tìm kiếm lân cận hình cầu (`query_ball_point`) để tính toán chính xác khoảng cách đến tiện ích gần nhất và mật độ tập trung tại các bán kính buffer 300m, 500m, 1,000m cho từng bất động sản, sinh ra bảng thuộc tính giàu thông tin gồm 40 trường dữ liệu.
3. **Phân tích bề mặt & Thống kê không gian nâng cao:**
   * **Nội suy IDW ($P=2$):** Thiết lập bề mặt giá liên tục phản ánh độ dốc suy giảm giá đất từ Đông sang Tây.
   * **Getis-Ord Gi*:** Định vị các cực tập trung giá trị với 26 điểm cực nóng (Hotspot 99% tại Hoàn Kiếm) và 32 điểm cực lạnh tương đối (Coldspot 99% tại Đống Đa).
   * **Local Moran's I (LISA):** Bóc tách chính xác các khu vực đồng nhất (19 điểm HH, 32 điểm LL) và phát hiện các điểm ngoại lai không gian đặc trưng (15 điểm HL mặt phố thương mại giữa vùng giá rẻ và 17 điểm LH nhà ngõ hẹp giữa vùng đất vàng).
   * **Truy vấn & Ghép không gian (Spatial Selection & Join):** Xác định 85.95% nhà ở đạt tiêu chuẩn giá cao hoặc gần trường học; 48.76% nhà nằm trong bán kính đi bộ 500m đến cơ sở y tế.

---

### 5.2. Các phát hiện cốt lõi về thị trường bất động sản nội đô
* **Quy luật phân cực không gian mạnh mẽ:** Thị trường nhà ở lõi Hà Nội chịu sự chi phối áp đảo bởi biến địa lý vĩ mô (`khu_vuc_Hoàn Kiếm` gia tăng tới ~202 triệu VNĐ/$m^2$ giá trị định giá). Lợi thế lịch sử, văn hóa và dịch vụ thương mại du lịch tạo ra một mức chênh lệch địa tô vị trí (Location Rent) vượt trội mà các yếu tố diện tích hay số tầng không thể bù đắp.
* **Tác động cận kề của cảnh quan mặt nước:** Trong số các tiện ích môi trường, khoảng cách tới hồ nước (`dist_water_lakes_m`) là biến cự ly duy nhất có ý nghĩa thống kê ($p = 0.047$), khẳng định giá trị kinh tế bền vững của vi khí hậu và cảnh quan hồ nước tại Hà Nội (mỗi mét xa hồ làm giảm giá trị khoảng 210,000 VNĐ/$m^2$).
* **Ngoại ứng thương mại của hình thái nhà mặt phố:** Yếu tố loại hình nhà ở đem lại phần bù giá trị xấp xỉ 88 triệu VNĐ/$m^2$, cho thấy thị trường sẵn sàng trả giá cao vượt trội cho năng lực sinh dòng tiền thương mại độc lập.

---

### 5.3. Đánh giá hạn chế kỹ thuật & Bài học thực nghiệm
* **Hiện tượng đa cộng tuyến khi kết hợp GIS với Kinh tế lượng:** Việc đưa trực tiếp các đại lượng kiểm định không gian cục bộ phái sinh từ chính giá bán ($Z$-score, $p$-value, $p_{FDR}$) vào mô hình OLS đã dẫn tới hiện tượng đa cộng tuyến trầm trọng (Condition Number vọt lên mức $8.26 \times 10^4$). Trong thực nghiệm kinh tế lượng, các chỉ số phân cụm chỉ nên đóng vai trò biến phân nhóm nhị phân (dummy variable) hoặc kiểm soát phương sai, không nên đưa trực tiếp dưới dạng số học liên tục. 
* **Hạn chế của mô hình hồi quy tuyến tính toàn cục (Global OLS):** OLS chỉ đạt $R^2 \approx 40.7\%$ và gặp hiện tượng "nén trần dự đoán" đối với nhóm nhà siêu sang do phân phối giá thực tế có độ lệch rất lớn. Mối quan hệ giữa giá đất và tiện ích đô thị vốn dĩ mang tính phi tuyến và biến thiên theo từng tiểu vùng địa phương.
* **Thách thức của phương pháp học máy (Random Forest) trên cỡ mẫu nhỏ:** Mô hình Random Forest bị sụt giảm $R^2$ nghiêm trọng trong quá trình kiểm định chéo 5-fold CV ($R^2 = 0.092$) do số lượng biến độc lập quá nhiều (36 biến) so với quy mô mẫu khiêm tốn ($N = 121$), dẫn tới hiện tượng quá khớp (overfitting) trên tập huấn luyện.

---

### 5.4. Đề xuất hướng phát triển tiếp theo
1. **Ứng dụng Hồi quy trọng số địa lý (GWR - Geographically Weighted Regression):** Cho phép hệ số hồi quy của từng tiện ích biến thiên linh hoạt theo tọa độ không gian $(u_i, v_i)$, giải quyết triệt để vấn đề bất đồng nhất không gian (Spatial Heterogeneity).
2. **Chuẩn hóa biến mục tiêu & Mở rộng thuộc tính:** Áp dụng mô hình hàm Semi-log ($\ln(Price)$) và bổ sung các trường dữ liệu vi mô thiết yếu: bề rộng ngõ trước nhà (m), chiều rộng mặt tiền, tình trạng pháp lý (sổ đỏ/sổ hồng/sổ chung) để triệt tiêu sai số ước lượng ở nhóm nhà giá thấp.
3. **Phát triển nền tảng WebGIS tương tác:** Đóng gói kết quả từ các file vector/raster sang GeoServer/Mapbox GL JS để xây dựng dashboard định giá bất động sản trực quan phục vụ người dùng cuối tra cứu trực tuyến.